# Link Adaptation: Results Visualization

This notebook compares the spectral efficiency achieved by agent-evolved link adaptation solutions and the OLLA (Outer Loop Link Adaptation) baseline.

For evaluation, we use a distinct set of SNR trajectories not seen during training.

## Imports

In [1]:
import os
import sys

_link_adaptation_dir = os.getcwd()
_eval_dir = os.path.join(_link_adaptation_dir, "eval")
os.chdir(_eval_dir)
sys.path.insert(0, _eval_dir)
from utils import get_bler, bler_2_mcs

import numpy as np
import pickle
from concurrent.futures import ProcessPoolExecutor, as_completed

from utils import get_bler, bler_2_mcs

from eval import (
    SINR_BOUNDS_LIN,
    BLER_TARGET,
    LinkAdaptation,
    run_la,
    NACK_REPORT_BATCH_SIZE,
    MCS_TO_SE,
    BLER_TARGET_TOLERANCE,
    BLER_SIGMOID_PARAMS,
)

## Evaluation utilities

The following utilities evaluate a link adaptation algorithm on each sequence in a dataset. 
For each sequence, they compute the average spectral efficiency and indicate whether the average block error rate (BLER) exceeds the target value.

To accelerate evaluation, SNR sequences are processed in parallel.

In [2]:
NUM_WORKERS = 10

def _load_dataset(dataset_path: str) -> np.ndarray:
    with open(dataset_path, "rb") as f:
        _, trajectories_gains = pickle.load(f)

    # Normalize gains to [0, 1] and scale to SINR bounds
    trajectories_gains = (
        trajectories_gains
        / np.max(trajectories_gains, axis=-1, keepdims=True)
    )
    trajectories_gains = (
        trajectories_gains * (SINR_BOUNDS_LIN[1] - SINR_BOUNDS_LIN[0])
        + SINR_BOUNDS_LIN[0]
    )

    return trajectories_gains

def _evaluate_single_scenario(scenario_idx: int, scenario: np.ndarray, mcs_selection: callable) -> dict:
    # Create MCS selection algorithm with agent's function
    la_algo = LinkAdaptation(
        BLER_TARGET,
        mcs_selection_func=mcs_selection,
    )

    # Run link adaptation simulation
    is_nack_hist, rate_hist, _, _, _ = run_la(
        la_algo,
        scenario,
        BLER_SIGMOID_PARAMS,
        nack_report_batch_size=NACK_REPORT_BATCH_SIZE,
        mcs_to_se=MCS_TO_SE,
    )

    # Compute metrics for this scenario
    long_term_bler = np.mean(is_nack_hist)
    success = long_term_bler < BLER_TARGET * (1.0 + BLER_TARGET_TOLERANCE)
    metric = np.mean(rate_hist)

    return {
        "scenario_idx": scenario_idx,
        "long_term_bler": long_term_bler,
        "success": success,
        "metric": metric,
    }


def evaluate_mcs_selection(mcs_selection: callable, dataset_path: str) -> (float, dict, dict, dict):

    # Run evaluation across all scenarios in parallel
    long_term_bler = {}
    success = {}
    metric = {}

    # Load dataset
    trajectories_gains = _load_dataset(dataset_path)

    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
        # Submit all scenarios for parallel evaluation
        futures = {
            executor.submit(_evaluate_single_scenario, idx, scenario, mcs_selection): idx
            for idx, scenario in enumerate(trajectories_gains)
        }

        # Collect results as they complete
        for future in as_completed(futures):
            result = future.result()
            scenario_idx = result["scenario_idx"]

            long_term_bler[scenario_idx] = result["long_term_bler"]
            success[scenario_idx] = result["success"]
            metric[scenario_idx] = result["metric"]

    # Generate output report
    average_metric = np.mean(list(metric.values()))

    return average_metric, success, metric, long_term_bler

Paths to the training and evaluation datasets.

These are two separate datasets with no overlapping SNR sequences:
- The training set is used to develop and optimize the agent's solution.
- The evaluation set is used in this notebook to assess the agent's solution against the baseline.

In [3]:
DATASET_EVALUATION_PATH = "data/trajectories_evaluation.pkl"
DATASET_TRAINING_PATH = "data/trajectories_training.pkl"

## Agent solution

Below are the best-performing solutions from each of the top three clusters.
In this instance, all top clusters have converged on the same approach.

### GPT-OSS 120B

In [4]:
class AgentEstimator_GPTOSS120B:

    def __init__(self):

        # Configuration constants (tuned for speed/accuracy trade‑off)
        self._N_PARTICLES = 100          # Number of particles – higher for accuracy, but keep moderate for speed.
        self._SINR_MIN = -10.0           # Minimum SINR (dB) considered.
        self._SINR_MAX = 30.0            # Maximum SINR (dB) considered.
        self._PROCESS_NOISE_STD = 0.5    # Standard deviation of SINR random‑walk process noise (dB).
        # We use a more aggressive percentile for SINR estimate.
        self._CONSERVATIVE_PERCENTILE = 0.2   # 20th percentile of the posterior.
        self._SINR_SAFETY_MARGIN = 0.5   # Safety margin (dB) subtracted from estimate to be extra conservative.

        # Persistent state across calls (module‑level globals)
        self.particles = None           # Shape: (N_PARTICLES,)
        self.weights = None             # Shape: (N_PARTICLES,)
        self._prev_len = 0              # Number of observations already processed.
        self._rng = None                # Random number generator instance.

    def _initialize(self):
        """Initialise the particle filter with a uniform prior.
        Called on the first invocation or when the history length shrinks.
        """
        self._rng = np.random.default_rng(42)
        self.particles = self._rng.uniform(self._SINR_MIN, self._SINR_MAX, size=self._N_PARTICLES)
        self.weights = np.full(self._N_PARTICLES, 1.0 / self._N_PARTICLES)
        self._prev_len = 0

    def _predict(self):
        """Random‑walk prediction step for all particles.
        The hidden SINR evolves as sinr_{t+1} = sinr_t + w_t, w_t ~ N(0, sigma^2).
        """
        self.particles += self._rng.normal(0.0, self._PROCESS_NOISE_STD, size=self._N_PARTICLES)
        self.particles = np.clip(self.particles, self._SINR_MIN, self._SINR_MAX)

    def _update(self, is_nack, mcs):
        """Bayesian update of particle weights using the latest ACK/NACK.

        Parameters
        ----------
        is_nack : int or bool
            1 (or True) if the last transmission resulted in a NACK, else 0/False.
        mcs : int
            MCS index used for the transmission that produced the feedback.
        """
        # Compute BLER for the given MCS across all particles.
        bler_vals = np.array([get_bler(s)[int(mcs)] for s in self.particles])
        # Likelihood of observation under each particle hypothesis.
        if is_nack:
            likelihood = bler_vals
        else:
            likelihood = 1.0 - bler_vals
        # Numerical stability: avoid zero likelihoods.
        eps = 1e-12
        likelihood = np.clip(likelihood, eps, 1.0)
        # Update particle weights.
        self.weights *= likelihood
        weight_sum = np.sum(self.weights)
        if weight_sum == 0 or np.isnan(weight_sum):
            # Degenerate case – reset to uniform distribution.
            self.weights = np.full(self._N_PARTICLES, 1.0 / self._N_PARTICLES)
        else:
            self.weights /= weight_sum
        # Resample if effective sample size is low.
        neff = 1.0 / np.sum(self.weights ** 2)
        if neff < self._N_PARTICLES / 2:
            self._systematic_resample()

    def _systematic_resample(self):
        """Systematic resampling to avoid particle degeneracy.
        After resampling, all particles have equal weight.
        """
        cumulative = np.cumsum(self.weights)
        step = 1.0 / self._N_PARTICLES
        start = self._rng.random() * step
        points = start + step * np.arange(self._N_PARTICLES)
        indexes = np.searchsorted(cumulative, points)
        self.particles = self.particles[indexes]
        self.weights = np.full(self._N_PARTICLES, 1.0 / self._N_PARTICLES)

    def _weighted_percentile(self, values: np.ndarray, weights: np.ndarray, percentile: float) -> float:
        """Return the weighted percentile of ``values``.

        Parameters
        ----------
        values : np.ndarray
            1‑D array of values.
        weights : np.ndarray
            Corresponding non‑negative weights (same shape as ``values``). They need
            not sum to one; the function normalises them internally.
        percentile : float
            Desired percentile in the range [0, 1] (e.g., 0.2 for the 20th percentile).
        """
        # Ensure inputs are numpy arrays.
        values = np.asarray(values)
        weights = np.asarray(weights)
        # Sort values and associated weights.
        idx = np.argsort(values)
        sorted_vals = values[idx]
        sorted_weights = weights[idx]
        # Normalise weights to sum to 1.
        total_weight = np.sum(sorted_weights)
        if total_weight == 0:
            # Avoid division by zero – fallback to unweighted percentile.
            return np.percentile(values, percentile * 100)
        normalized_weights = sorted_weights / total_weight
        # Compute cumulative sum of weights.
        cum_weights = np.cumsum(normalized_weights)
        # Find the first index where cumulative weight exceeds the percentile.
        target = percentile
        index = np.searchsorted(cum_weights, target, side='right')
        # Clamp index to valid range.
        index = min(max(index, 0), len(sorted_vals) - 1)
        return float(sorted_vals[index])

    def _conservative_mcs(self, bler_target: float) -> int:
        """Select a conservative MCS based on a low‑percentile SINR estimate.
        This is used as a safe fallback when no candidate satisfies the BLER
        constraint.
        """
        est_sinr = self._weighted_percentile(self.particles, self.weights, self._CONSERVATIVE_PERCENTILE)
        est_sinr = est_sinr - self._SINR_SAFETY_MARGIN
        est_sinr = max(min(est_sinr, self._SINR_MAX), self._SINR_MIN)
        bler_vec = get_bler(est_sinr)
        return int(bler_2_mcs(bler_vec, bler_target))

    def _max_mcs(self) -> int:
        """Return the maximum MCS index supported by the utilities.
        We query the helper at a high SINR; if that fails we fall back to a typical
        upper bound.
        """
        try:
            return int(bler_2_mcs(get_bler(30.0), 0.1))
        except Exception:
            return 28

    def __call__(self, is_nack_hist: np.ndarray,
                    mcs_ackned_hist: np.ndarray,
                    bler_target: float) -> int:
        """Entry point for the evaluation harness.

        This implementation follows the *Monte‑Carlo look‑ahead* approach:
        1. Maintain a particle filter that tracks the posterior distribution of the
        hidden SINR.
        2. For each possible MCS, compute the *expected* spectral efficiency of the
        next transmission (weighted by the particle posterior) while ensuring the
        average BLER stays below the target.
        3. Add a short look‑ahead horizon (future steps) using a safe policy that
        always picks the highest MCS meeting the BLER target for the predicted
        SINR.
        4. Return the first‑step MCS that maximises this expected cumulative reward.
        """
        # Convert inputs to NumPy arrays (ensure correct dtype).
        is_nack_hist = np.asarray(is_nack_hist, dtype=int)
        mcs_ackned_hist = np.asarray(mcs_ackned_hist, dtype=int)

        # Initialise on first call or if we have no history.
        if self.particles is None or len(is_nack_hist) == 0:
            self._initialize()
            return self._conservative_mcs(bler_target)

        # Detect scenario reset (history length decreased).
        if len(is_nack_hist) < self._prev_len:
            self._initialize()

        # Process only the new observations since the last call.
        for idx in range(self._prev_len, len(is_nack_hist)):
            self._predict()
            self._update(is_nack_hist[idx], mcs_ackned_hist[idx])
        self._prev_len = len(is_nack_hist)

        # Determine the maximum MCS index supported by the utilities.
        max_mcs = self._max_mcs()
        num_mcs = max_mcs + 1

        # ---------------------------------------------------------------------
        # 1️⃣ Compute the BLER matrix for all particles (shape: N_PARTICLES x num_mcs).
        # ---------------------------------------------------------------------
        # This is the most expensive part but we need it only once per call.
        bler_matrix = np.array([get_bler(s)[:num_mcs] for s in self.particles])

        # ---------------------------------------------------------------------
        # 2️⃣ Expected immediate reward and one‑step look‑ahead for each candidate MCS.
        # ---------------------------------------------------------------------
        # Weighted average BLER per candidate.
        avg_bler = np.dot(self.weights, bler_matrix)  # shape: (num_mcs,)
        # Determine which candidates satisfy the BLER target on average.
        feasible = avg_bler <= bler_target

        if not np.any(feasible):
            # No candidate respects the BLER target – fall back to a conservative
            # estimate based on a low‑percentile SINR.
            return self._conservative_mcs(bler_target)

        # Pre‑compute a list of candidate indices that are feasible.
        feasible_indices = np.where(feasible)[0]
        # We'll store the total expected reward for each feasible candidate.
        total_rewards = np.empty_like(feasible_indices, dtype=float)

        # Helper: compute the safe MCS (conservative) given a weight distribution.
        def _safe_mcs_for_weights(w_dist):
            est_sinr = self._weighted_percentile(self.particles, w_dist, self._CONSERVATIVE_PERCENTILE)
            est_sinr = est_sinr - self._SINR_SAFETY_MARGIN
            est_sinr = max(min(est_sinr, self._SINR_MAX), self._SINR_MIN)
            bler_vec = get_bler(est_sinr)
            mcs = int(bler_2_mcs(bler_vec, bler_target))
            return max(0, min(mcs, max_mcs))

        # Iterate over feasible candidates and compute the look‑ahead reward.
        for idx, cand_mcs in enumerate(feasible_indices):
            # BLER values for this candidate across particles.
            bler_cand = bler_matrix[:, cand_mcs]
            # Immediate expected reward for this candidate.
            immediate = np.dot(self.weights, (1.0 - bler_cand) * cand_mcs)
            # Probability of ACK/NACK for this candidate.
            p_ack = np.dot(self.weights, (1.0 - bler_cand))
            p_nack = 1.0 - p_ack
            # Expected future reward after an ACK.
            if p_ack > 0:
                w_ack = self.weights * (1.0 - bler_cand) / p_ack
                safe_mcs_ack = _safe_mcs_for_weights(w_ack)
                # Use pre‑computed BLER matrix for the safe MCS.
                bler_safe_ack = bler_matrix[:, safe_mcs_ack]
                future_ack = np.dot(w_ack, (1.0 - bler_safe_ack) * safe_mcs_ack)
            else:
                future_ack = 0.0
            # Expected future reward after a NACK.
            if p_nack > 0:
                w_nack = self.weights * bler_cand / p_nack
                safe_mcs_nack = _safe_mcs_for_weights(w_nack)
                bler_safe_nack = bler_matrix[:, safe_mcs_nack]
                future_nack = np.dot(w_nack, (1.0 - bler_safe_nack) * safe_mcs_nack)
            else:
                future_nack = 0.0
            # Combine immediate and expected future rewards.
            total_rewards[idx] = immediate + p_ack * future_ack + p_nack * future_nack

        # Choose the candidate with the highest total expected reward.
        best_feasible_idx = feasible_indices[np.argmax(total_rewards)]
        return int(best_feasible_idx)

### GPT 5.4

In [5]:
class AgentSolution_GPT54:

    def __init__(self):

        self._GRID = np.linspace(-12.0, 30.0, 169)
        self._BLER_TABLE = np.array([get_bler(float(s)) for s in self._GRID])
        self._DX = float(self._GRID[1] - self._GRID[0])
        self._N_MCS = self._BLER_TABLE.shape[1]
        self._SIG = 0.52
        self._RAD = max(1, int(np.ceil(2.5 * self._SIG / self._DX)))
        self._OFF = np.arange(-self._RAD, self._RAD + 1)
        self._KERNEL = np.exp(-0.5 * ((self._OFF * self._DX) / self._SIG) ** 2)
        self._KERNEL /= self._KERNEL.sum()
        self._PRIOR_MEAN = 9.0
        self._PRIOR_STD = 6.5
        self._PRIOR = np.exp(-0.5 * ((self._GRID - self._PRIOR_MEAN) / self._PRIOR_STD) ** 2)
        self._PRIOR /= self._PRIOR.sum()

    def _posterior_trimmed_mean(self,y, m, max_hist=64, forget_mix=0.006, q_lo=0.15, q_hi=0.85):
        probs = self._PRIOR.copy()
        start = max(0, len(y) - max_hist)
        for nack, mcs in zip(y[start:], m[start:]):
            probs = np.convolve(probs, self._KERNEL, mode='same')
            probs = (1.0 - forget_mix) * probs + forget_mix / probs.size
            p_nack = np.clip(self._BLER_TABLE[:, int(mcs)], 1e-7, 1.0 - 1e-7)
            probs *= p_nack if int(nack) else (1.0 - p_nack)
            s = probs.sum()
            probs = self._PRIOR.copy() if (s <= 0.0 or not np.isfinite(s)) else (probs / s)
        cdf = np.cumsum(probs)
        lo = int(np.searchsorted(cdf, q_lo))
        hi = int(np.searchsorted(cdf, q_hi))
        lo = min(max(lo, 0), len(self._GRID) - 1)
        hi = min(max(hi, lo), len(self._GRID) - 1)
        w = probs[lo:hi + 1]
        x = self._GRID[lo:hi + 1]
        sw = float(w.sum())
        if sw <= 0.0 or not np.isfinite(sw):
            idx = int(np.searchsorted(cdf, 0.5))
            idx = min(max(idx, 0), len(self._GRID) - 1)
            return float(self._GRID[idx])
        return float(np.dot(w, x) / sw)

    def __call__(self, is_nack_hist: np.ndarray, mcs_ackned_hist: np.ndarray, bler_target: float) -> int:
        y = np.asarray(is_nack_hist, dtype=int)
        m = np.asarray(mcs_ackned_hist, dtype=int)
        n = min(len(y), len(m))
        target = float(bler_target)
        if n == 0:
            return int(np.clip(bler_2_mcs(get_bler(7.0), target), 0, self._N_MCS - 1))
        y = y[-n:]
        m = np.clip(m[-n:], 0, self._N_MCS - 1)
        base = self._posterior_trimmed_mean(y, m, max_hist=64, forget_mix=0.006, q_lo=0.15, q_hi=0.85)
        step = 0.0237
        step_down = step * (1.0 - target) / max(target, 1e-9)
        offset = step * float(np.sum(y == 0)) - step_down * float(np.sum(y == 1))
        offset = float(np.clip(offset, -2.94, 0.25))
        warm = -0.95 if n < 5 else (-0.35 if n < 10 else 0.0)
        est = float(np.clip(base + offset + warm, self._GRID[0], self._GRID[-1]))
        return int(np.clip(bler_2_mcs(get_bler(est), target), 0, self._N_MCS - 1))

## Baseline: OLLA

In [6]:
class BaselineOLLA:

    def __init__(self, delta_down: float = 1.0):
        self.delta_down = delta_down

    def __call__(self, is_nack_hist: np.ndarray,
                    mcs_ackned_hist: np.ndarray,
                    bler_target: float) -> int:

        # OLLA parameters
        delta_down = self.delta_down
        delta_up = delta_down * bler_target / (1.0 - bler_target)

        # Count ACKs and NACKs
        num_nack = np.sum(is_nack_hist)
        num_ack = len(is_nack_hist) - num_nack

        # OLLA: update SINR estimate from initial value
        sinr_est = delta_up * num_ack - delta_down * num_nack

        # ILLA: pick the highest MCS whose BLER is below the target
        bler = get_bler(sinr_est)
        mcs = bler_2_mcs(bler, bler_target)

        return mcs

    def optimize_delta_down(self, bler_target: float, dataset_path: str) -> float:
        # Grid of delta_down values
        delta_down_values = np.linspace(0.5, 4.0, 20)
        best_average_metric = -np.inf
        best_delta_down = None
        for i, delta_down in enumerate(delta_down_values):
            self.delta_down = delta_down
            average_metric, success, _, _ = evaluate_mcs_selection(self, dataset_path)
            all_success = np.all(list(success.values()))
            if average_metric > best_average_metric and all_success:
                best_average_metric = average_metric
                best_delta_down = delta_down
            print(f"{i} / {len(delta_down_values)}: {best_delta_down}, {best_average_metric:.4f}", end="\r")
        self.delta_down = best_delta_down

To ensure fairness, the OLLA baseline parameter is optimized to maximize spectral efficiency on the training set.

In [7]:
baseline_mcs_selection = BaselineOLLA()
baseline_mcs_selection.optimize_delta_down(BLER_TARGET, DATASET_TRAINING_PATH)


# Evaluation

In [8]:
METRICS = {}
SUCCESS = {}
AVERAGE_METRICS = {}

In [9]:
### GPT-OSS 120B
AVERAGE_METRICS['agent-gpt-oss-120b'], SUCCESS['agent-gpt-oss-120b'], METRICS['agent-gpt-oss-120b'], _ = evaluate_mcs_selection(AgentEstimator_GPTOSS120B(), DATASET_EVALUATION_PATH)

### GPT 5.4
AVERAGE_METRICS['agent-gpt-5.4'], SUCCESS['agent-gpt-5.4'], METRICS['agent-gpt-5.4'], _ = evaluate_mcs_selection(AgentSolution_GPT54(), DATASET_EVALUATION_PATH)

### Baseline
AVERAGE_METRICS['baseline'], SUCCESS['baseline'], METRICS['baseline'], _ = evaluate_mcs_selection(baseline_mcs_selection, DATASET_EVALUATION_PATH)

In [10]:
col_w = 20

# Header
print(f"{'Algorithm':<{col_w}} {'Avg Metric (bps/Hz)':>20} {'Success (count)':>18} {'vs Baseline':>13}")
print("-" * (col_w + 20 + 18 + 13 + 3))

baseline_metric = AVERAGE_METRICS["baseline"]

for name, avg_metric in AVERAGE_METRICS.items():
    successes = list(SUCCESS[name].values())
    num_success = sum(successes)
    total = len(successes)
    success_str = f"{num_success}/{total}"
    diff = (avg_metric - baseline_metric) / baseline_metric * 100.0
    diff_str = f"{diff:+.2f}%" if name != "baseline" else "---"
    print(f"{name:<{col_w}} {avg_metric:>20.4f} {success_str:>18} {diff_str:>13}")

Algorithm             Avg Metric (bps/Hz)    Success (count)   vs Baseline
--------------------------------------------------------------------------
agent-gpt-oss-120b                 3.3976              50/50        -0.64%
agent-gpt-5.4                      3.5370              50/50        +3.43%
baseline                           3.4196              50/50           ---
